# Interrupted quadratic-Bezier comparison — recovery implementation validation

This notebook validates the guarded recovery implementation without mounting Google Drive, accessing the preserved `.incomplete` directory, creating scientific outputs, revealing metrics, opening images, resuming execution, or starting a new comparison. Use a fresh CPU runtime and run all six code cells in order. Return the downloaded JSON report, pytest log, and unauthorized-probe log. A passing report does not authorize recovery execution.

In [ ]:
from pathlib import Path
import shutil
import subprocess

REPO_DIR = Path('/content/latent-stroke-dynamics')
REPO_URL = 'https://github.com/Navid111/latent-stroke-dynamics.git'
BRANCH = 'quadratic-bezier-extension'
RECOVERY_IMPLEMENTATION_COMMIT = '46e0c6396f0425ed84812e8fbeef9ed675ef53e9'
FROZEN_RUNNER_COMMIT = '398a2bfb7bd65ed8b4bbc93fb8cc05564f7f3c1b'

if Path('/content/drive/MyDrive').exists():
    raise RuntimeError('STOP: Google Drive is mounted. Use a fresh runtime for no-output validation.')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
clone = subprocess.run(
    ['git', 'clone', '--quiet', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)],
    capture_output=True,
    text=True,
)
if clone.returncode != 0:
    print(clone.stdout)
    print(clone.stderr)
    raise RuntimeError('Public repository clone failed.')
subprocess.run(
    ['git', 'checkout', '--quiet', '--detach', RECOVERY_IMPLEMENTATION_COMMIT],
    cwd=REPO_DIR,
    check=True,
)
observed_commit = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True
).strip()
assert observed_commit == RECOVERY_IMPLEMENTATION_COMMIT
assert subprocess.check_output(
    ['git', 'status', '--short'], cwd=REPO_DIR, text=True
).strip() == ''
print('CELL 1 COMPLETE — EXACT UNAUTHORIZED RECOVERY IMPLEMENTATION PINNED')
print('recovery implementation:', RECOVERY_IMPLEMENTATION_COMMIT)
print('frozen runner:', FROZEN_RUNNER_COMMIT)

In [ ]:
import re
import subprocess
import time
from pathlib import Path

subprocess.run(
    ['python', '-m', 'pip', 'install', '-q', '-e', '.'],
    cwd=REPO_DIR,
    check=True,
)
test_started = time.monotonic()
test_run = subprocess.run(
    ['python', '-m', 'pytest', '-q'],
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
)
test_duration_seconds = time.monotonic() - test_started
PYTEST_LOG = Path('/content/quadratic_bezier_recovery_pytest.txt')
PYTEST_LOG.write_text(test_run.stdout + test_run.stderr, encoding='utf-8')
print(test_run.stdout)
if test_run.stderr:
    print(test_run.stderr)
assert test_run.returncode == 0, 'The complete test suite failed; recovery remains unauthorized.'
match = re.search(r'(\d+) passed', test_run.stdout + test_run.stderr)
assert match is not None, 'Could not parse pytest pass count.'
pytest_passed = int(match.group(1))
assert pytest_passed == 217, pytest_passed
print('CELL 2 COMPLETE — COMPLETE 217-TEST SUITE PASSED')

In [ ]:
import json
import subprocess

PLAN_PATH = REPO_DIR / 'configs/quadratic-bezier-interrupted-recovery-plan-2026-09-05.json'
FREEZE_PATH = REPO_DIR / 'configs/quadratic-bezier-target-freeze-2026-09-04.json'
ORIGINAL_AUTHORIZATION_PATH = REPO_DIR / 'configs/quadratic-bezier-execution-authorization-2026-09-04.json'
validation_run = subprocess.run(
    [
        'python',
        'run_quadratic_bezier_recovery.py',
        '--validate-only',
        '--repo-root', str(REPO_DIR),
        '--plan', str(PLAN_PATH),
        '--freeze', str(FREEZE_PATH),
        '--expected-head', RECOVERY_IMPLEMENTATION_COMMIT,
    ],
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
)
print(validation_run.stdout)
if validation_run.stderr:
    print(validation_run.stderr)
assert validation_run.returncode == 0, 'Recovery validation failed.'
validation = json.loads(validation_run.stdout)
assert validation['status'] == 'quadratic_bezier_recovery_implementation_valid_no_outputs_unauthorized'
assert validation['observed_recovery_commit'] == RECOVERY_IMPLEMENTATION_COMMIT
assert validation['frozen_runner_commit'] == FROZEN_RUNNER_COMMIT
assert validation['preserved_completed_run_count'] == 17
assert validation['quarantined_partial_run_count'] == 1
assert validation['missing_run_count'] == 19
assert validation['recovery_execution_authorized'] is False
assert validation['interrupted_output_accessed'] is False
assert validation['interrupted_output_modified'] is False
assert validation['recovery_output_created'] is False
assert validation['comparative_metrics_revealed'] is False
print('CELL 3 COMPLETE — RECOVERY VALIDATION PASSED WITHOUT DRIVE ACCESS')

In [ ]:
from pathlib import Path
import subprocess

PROBE_OUTPUT = Path('/content/quadratic-bezier-fixed-comparison-v1')
PROBE_INCOMPLETE = Path('/content/quadratic-bezier-fixed-comparison-v1.incomplete')
MISSING_RECOVERY_AUTHORIZATION = Path('/content/recovery-authorization-does-not-exist.json')
assert not PROBE_OUTPUT.exists()
assert not PROBE_INCOMPLETE.exists()
assert not MISSING_RECOVERY_AUTHORIZATION.exists()
probe = subprocess.run(
    [
        'python',
        'run_quadratic_bezier_recovery.py',
        '--execute-recovery',
        '--repo-root', str(REPO_DIR),
        '--plan', str(PLAN_PATH),
        '--freeze', str(FREEZE_PATH),
        '--output-dir', str(PROBE_OUTPUT),
        '--original-authorization', str(ORIGINAL_AUTHORIZATION_PATH),
        '--recovery-authorization', str(MISSING_RECOVERY_AUTHORIZATION),
        '--recovery-implementation-commit', RECOVERY_IMPLEMENTATION_COMMIT,
    ],
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
)
UNAUTHORIZED_PROBE_LOG = Path('/content/quadratic_bezier_recovery_unauthorized_probe.txt')
UNAUTHORIZED_PROBE_LOG.write_text(probe.stdout + probe.stderr, encoding='utf-8')
assert probe.returncode != 0, 'Unauthorized recovery unexpectedly succeeded.'
assert not PROBE_OUTPUT.exists()
assert not PROBE_INCOMPLETE.exists()
print('CELL 4 COMPLETE — UNAUTHORIZED RECOVERY REFUSED BEFORE OUTPUT CREATION')

In [ ]:
from hashlib import sha256
import json
from pathlib import Path

report = {
    'status': 'quadratic_bezier_recovery_implementation_valid_no_outputs_unauthorized',
    'protocol_id': 'quadratic_bezier_extension_v1',
    'recovery_implementation_commit': RECOVERY_IMPLEMENTATION_COMMIT,
    'frozen_runner_commit': FROZEN_RUNNER_COMMIT,
    'recovery_plan_sha256': sha256(PLAN_PATH.read_bytes()).hexdigest(),
    'target_freeze_sha256': sha256(FREEZE_PATH.read_bytes()).hexdigest(),
    'pytest_passed': pytest_passed,
    'pytest_duration_seconds': test_duration_seconds,
    'validation': validation,
    'unauthorized_recovery_refused': True,
    'unauthorized_output_created': False,
    'google_drive_mounted': False,
    'interrupted_output_accessed': False,
    'interrupted_output_modified': False,
    'comparative_outputs_viewed': False,
    'comparative_metrics_revealed': False,
    'recovery_execution_authorized': False,
    'training_performed': False,
    'learned_model_used': False,
    'closed_experiments_changed': False,
}
REPORT_PATH = Path('/content/quadratic_bezier_recovery_validation_report.json')
REPORT_PATH.write_text(
    json.dumps(report, indent=2, sort_keys=True) + '\n', encoding='utf-8'
)
print('CELL 5 COMPLETE — VALIDATION REPORT WRITTEN')
print('status:', report['status'])
print('pytest passed:', pytest_passed)
print('recovery authorized:', report['recovery_execution_authorized'])

In [ ]:
from google.colab import files

files.download(str(REPORT_PATH))
files.download(str(PYTEST_LOG))
files.download(str(UNAUTHORIZED_PROBE_LOG))
print('CELL 6 COMPLETE — THREE VALIDATION FILES DOWNLOADED')
print('Return all three files. Do not mount Drive or run recovery.')